In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/train.csv')
train.shape

Before doing any analysis, I wanted to understand the basic shape of the dataset: how many rows and columns it has, what the columns are called, and what a single row actually represents. The train set has 6,818 rows and 13 columns. Each row represents one product-store pairing, a specific product sold at a specific store, along with the total sales that combination generated. This matters because the task isn't "predict sales for a product" or "predict sales for a store" in isolation, it's predicting sales for a specific product-store combination, which is why both product-level and store-level columns appear together in every row.

In [ ]:
train.head()

In [ ]:
train.isnull().sum()

I checked every column for missing values rather than assuming the data was complete. Two columns had gaps: product_weight_kg (~18% missing) and store_size (~28% missing). Given how large these percentages are, I decided against simply dropping the affected rows; doing so could have removed up to 46% of the dataset in the worst case, discarding otherwise complete, useful information in every other column just because of one missing field. Instead, I planned to fill these gaps later (imputation) rather than delete data.

In [ ]:
train.dtypes

I checked each categorical column's actual values using .unique() rather than assuming their meaning from the column name alone. This distinguished two types of categorical data:

Ordinal (has a real order): store_size (Small/Medium/Large), store_location_tier (Tier_1/2/3), store_format (Corner Shop → Flagship Hypermarket, per the data dictionary's listed order), fat_content (Low Fat/Regular).
Nominal (no real order): product_category.

I also discovered that product_category had 47 unique values instead of the expected ~16, due to inconsistent capitalization (e.g. 'Canned', 'CANNED', 'canned' all appearing as separate categories). This is a real data quality issue (confirmed intentional by the dataset's own data dictionary) that would have distorted any category-level analysis if left unaddressed — for example, initial (uncleaned) groupby averages varied by 100+ units between different-cased versions of the same true category

In [ ]:
train['store_format'].unique()

In [ ]:
train['fat_content'].unique()

In [ ]:
train['store_location_tier'].unique()

In [ ]:
train['product_category'].unique()

In [ ]:
# Get summary statistics (count, mean, spread, min/max) for all numeric columns
train.describe()

I compared total_sales's mean (2174.76) to its median (1790.89) and found a meaningful gap, suggesting a right-skewed distribution; most product-store combinations generate modest sales, with a smaller number of high-sales outliers pulling the average upward. I confirmed this visually with a histogram, which showed a long tail extending toward the higher sales values. This is a normal, expected shape for real-world sales data, not a data error.

In [ ]:
# Import matplotlib's plotting module, nicknamed plt by convention
import matplotlib.pyplot as plt

# Plot a histogram of total_sales, split into 30 bins
train['total_sales'].hist(bins=30)

# Label the chart so it's readable
plt.xlabel('Total Sales')
plt.ylabel('Number of rows')
plt.title('Distribution of Total Sales')
plt.show()

Rather than assuming which features would matter, I checked each one against total_sales directly:

Categorical columns (via group averages): store_format, store_size, store_location_tier, and product_category (after cleaning) all showed meaningful variation in average sales across their categories — suggesting real predictive value. Some patterns were counter-intuitive (e.g. Tier_1 — major urban centers — actually had the lowest average sales, not the highest), reinforcing the importance of checking real data rather than assuming.
Numeric columns (via correlation): product_price had the strongest relationship with total_sales (correlation ≈ 0.57), while product_weight_kg and store_age_years showed almost no linear relationship (≈0.02–0.04). A scatter plot of price vs. sales confirmed a real but noisy upward trend, not a perfect line — indicating price alone doesn't fully determine sales

In [ ]:
# Group rows by store_format, then find the average total_sales within each group
train.groupby('store_format')['total_sales'].mean()

In [ ]:
train.groupby('store_size')['total_sales'].mean()

In [ ]:
train.groupby('store_location_tier')['total_sales'].mean()

In [ ]:
train.groupby('product_category')['total_sales'].mean()

In [ ]:
# Overwrite product_category with a lowercase version of itself, to fix the case-inconsistency mess
train['product_category'] = train['product_category'].str.lower()

In [ ]:
train['product_category'].unique()

In [ ]:
train.groupby('product_category')['total_sales'].mean()

In [ ]:
# Calculate correlation between total_sales and every other numeric column
train.corr(numeric_only=True)['total_sales']

In [ ]:
# Scatter plot: product_price on x-axis, total_sales on y-axis
plt.scatter(train['product_price'], train['total_sales'], alpha=0.3)

plt.xlabel('Product Price')
plt.ylabel('Total Sales')
plt.title('Product Price vs Total Sales')
plt.show()

* product_weight_kg (numeric, ~18% missing): checked whether the column was skewed by comparing mean (12.85) to median (12.65) — they were close, indicating a roughly symmetric distribution, so I used mean imputation.
* store_size (categorical, ~28% missing): filled missing values with the mode (most frequent category, "Medium"), since averaging text categories isn't meaningful.

Both train and test were filled using statistics calculated only from train, to avoid data leakage (letting information from the test set influence preprocessing decisions).

In [ ]:
# Fill missing product_weight_kg values with the column's mean
train['product_weight_kg'] = train['product_weight_kg'].fillna(train['product_weight_kg'].mean())

In [ ]:
train['product_weight_kg'].isnull().sum()

In [ ]:
# Find which store_size category appears most often
train['store_size'].mode()

In [ ]:
# Fill missing store_size values with the most frequent category (Medium)
train['store_size'] = train['store_size'].fillna(train['store_size'].mode()[0])

In [ ]:
train['store_size'].isnull().sum()

In [ ]:
# Final check: confirm zero missing values across the entire dataset
train.isnull().sum()

In [ ]:
# Load the test dataset
test = pd.read_csv('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/test.csv')

# Check its shape and columns
print(test.shape)
print(test.columns.tolist())

In [ ]:
# Standardize test's product_category casing, same as we did for train
test['product_category'] = test['product_category'].str.lower()

In [ ]:
test['product_category'].unique()

In [ ]:
# Check missing values in test
test.isnull().sum()

In [ ]:
# Fill test's missing product_weight_kg using TRAIN's mean (not test's own mean)
test['product_weight_kg'] = test['product_weight_kg'].fillna(train['product_weight_kg'].mean())

In [ ]:
# Fill test's missing store_size using TRAIN's mode (not test's own mode)
test['store_size'] = test['store_size'].fillna(train['store_size'].mode()[0])

In [ ]:
test.isnull().sum()

In [ ]:
# Manually define the order we want: Small=0, Medium=1, Large=2
size_order = {'Small': 0, 'Medium': 1, 'Large': 2}

# Map store_size text values to these numbers, in both train and test
train['store_size'] = train['store_size'].map(size_order)
test['store_size'] = test['store_size'].map(size_order)

In [ ]:
train['store_size'].unique()

In [ ]:
train['store_size'].dtype

In [ ]:
train['store_size'].unique()

In [ ]:
test['store_size'].unique()

In [ ]:
store_location_tier = {'Tier_1' : 2, 'Tier_2': 1, 'Tier_3': 0}

train['store_location_tier'] = train ['store_location_tier'].map(store_location_tier)
test['store_location_tier'] = test ['store_location_tier'].map(store_location_tier)

In [ ]:
train['store_location_tier'].unique()

In [ ]:
train['store_location_tier'].dtype

In [ ]:
train['store_location_tier'].unique()

In [ ]:
test['store_location_tier'].unique()

In [ ]:
# Define the order for store_format: Corner Shop (smallest) to Flagship Hypermarket (biggest)
store_format = {'Corner Shop': 0, 'Standard Supermarket': 1, 'Superstore': 2, 'Flagship Hypermarket': 3}

# Apply the mapping to both train and test
train['store_format'] = train['store_format'].map(store_format)
test['store_format'] = test['store_format'].map(store_format)

# Define the order for fat_content: Low Fat = 0, Regular = 1
fat_content = {'Low Fat': 0, 'Regular': 1}

# Apply the mapping to both train and test
train['fat_content'] = train['fat_content'].map(fat_content)
test['fat_content'] = test['fat_content'].map(fat_content)

In [ ]:
train['store_format'].unique()
test['store_format'].unique()
train['fat_content'].unique()
test['fat_content'].unique()

In [ ]:
train['store_format'].unique()

In [ ]:
test['store_format'].unique()

In [ ]:
train['fat_content'].unique()

In [ ]:
test['fat_content'].unique()

In [ ]:
train['store_size'].unique()

In [ ]:
train['store_location_tier'].unique()

In [ ]:
train['store_format'].unique()

In [ ]:
train['fat_content'].unique()

In [ ]:
test['store_size'].unique()

In [ ]:
test['store_location_tier'].unique()

In [ ]:
test['store_format'].unique()


In [ ]:
test['fat_content'].unique()

Models require numeric input, so I converted each categorical column based on whether it had a natural order:

Ordinal encoding (manual integer mapping, preserving real-world order) for store_size, store_location_tier, store_format, and fat_content.
One-hot encoding (pd.get_dummies) for product_category (16 categories) and store_code (only 10 unique stores — low enough cardinality to be useful, unlike product_code).

I deliberately avoided one-hot encoding product_code (1,555 unique values) and id (unique per row), since both have very high cardinality — encoding them would create hundreds/thousands of mostly-empty columns carrying little to no generalizable signal, and id in particular is a pure identifier with zero real predictive meaning.

In [ ]:
# One-hot encode product_category for train
train = pd.get_dummies(train, columns=['product_category'], prefix='cat')

# One-hot encode product_category for test
test = pd.get_dummies(test, columns=['product_category'], prefix='cat')

In [ ]:
train.columns.tolist()

In [ ]:
test.columns.tolist()

In [ ]:
train.shape
test.shape

In [ ]:
train.shape

In [ ]:
train['product_code'].nunique()

In [ ]:
train['store_code'].nunique()

In [ ]:
# One-hot encode store_code for train
train = pd.get_dummies(train, columns=['store_code'], prefix='store')

# One-hot encode store_code for test
test = pd.get_dummies(test, columns=['store_code'], prefix='store')

In [ ]:
train.columns.tolist()

In [ ]:
test.columns.tolist()

In [ ]:
train.shape
test.shape

In [ ]:
train.shape

In [ ]:
# Save test's id column separately, for use in the final submission file later
test_ids = test['id']

# Drop id and product_code from train (id has no predictive value; product_code has high cardinality)
train = train.drop(columns=['id', 'product_code'])

# Drop the same columns from test, for consistency
test = test.drop(columns=['id', 'product_code'])

In [ ]:
train.shape
test.shape
test_ids.head()

In [ ]:
train.shape

In [ ]:
test.shape

I split the training data 80/20 into a training set and a held-out validation set, to evaluate model performance on data the model never saw during training. This guards against overfitting — a model that performs well only on data it has memorized, rather than learning generalizable patterns. total_sales was separated out as the target (y), with all other columns as features (X), ensuring the target never leaked into the model's inputs.

In [ ]:
# X = all our feature columns (everything except the target)
X = train.drop(columns=['total_sales'])

# y = just the target column, what we're trying to predict
y = train['total_sales']

In [ ]:
X.head()
y.head()

In [ ]:
# Import the splitting tool from scikit-learn
from sklearn.model_selection import train_test_split

# Split X and y into training (80%) and validation (20%) portions
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape
X_val.shape
y_train.shape
y_val.shape

In [ ]:
X_train.shape

In [ ]:
X_val.shape

In [ ]:
y_train.shape

I compared three models on the same train/validation split, using RMSE and R² as evaluation metrics:

Model	Validation RMSE	R²
Linear Regression	1122.56	0.572
Random Forest (default)	1109.10	0.582

Random Forest outperformed Linear Regression on both metrics, likely because the price-vs-sales relationship (and others) isn't purely linear — Random Forest can capture more complex, non-linear patterns.

To confirm this wasn't due to a lucky single split, I ran 5-fold cross-validation on both models:

Linear Regression: 1129.73 average RMSE
Random Forest: 1117.26 average RMSE

Random Forest's advantage held up under cross-validation, confirming it as the stronger choice.

I later also tested XGBoost, a gradient boosting model that builds trees sequentially, each correcting the previous trees' errors. An untuned XGBoost initially scored worse than Random Forest (1127.09 vs 1109.10), but after tuning (see below), it became the best-performing model overall.

In [ ]:
# Import the Linear Regression model from scikit-learn
from sklearn.linear_model import LinearRegression

# Create the model (untrained, just a blank formula-searcher for now)
model = LinearRegression()

# Train it: let it study X_train and y_train to find the best weights
model.fit(X_train, y_train)

In [ ]:
# Use the trained model to predict total_sales for the validation set
y_pred = model.predict(X_val)

In [ ]:
# Compare the first 5 predictions to the first 5 real answers
print(y_pred[:5])
print(y_val[:5].values)

In [ ]:
# Import RMSE calculation tools from scikit-learn
from sklearn.metrics import mean_squared_error
import numpy as np

# Calculate RMSE: root mean squared error between predictions and actual values
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(rmse)

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_val, y_pred)
print(r2)

In [ ]:
# Import Random Forest Regressor from scikit-learn
from sklearn.ensemble import RandomForestRegressor

# Create the model (100 trees is a common, reasonable default)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train it on the exact same training data as before
rf_model.fit(X_train, y_train)

In [ ]:
# Predict on validation set
rf_pred = rf_model.predict(X_val)

# Calculate RMSE and R²
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
rf_r2 = r2_score(y_val, rf_pred)

print(rf_rmse)
print(rf_r2)

In [ ]:
# Import the cross-validation tool
from sklearn.model_selection import cross_val_score

# Run 5-fold cross-validation using our Random Forest model, scoring by RMSE
cv_scores = cross_val_score(rf_model, X, y, cv=5, scoring='neg_root_mean_squared_error')

# Convert to positive RMSE values (sklearn returns them as negative by convention) and view all 5
print(-cv_scores)

# Average across all 5 folds
print((-cv_scores).mean())

In [ ]:
# Import Linear Regression again (in case it's not already available in this session)
from sklearn.linear_model import LinearRegression

# Create a fresh Linear Regression model
lr_model = LinearRegression()

# Run 5-fold cross-validation on Linear Regression too
lr_cv_scores = cross_val_score(lr_model, X, y, cv=5, scoring='neg_root_mean_squared_error')

print(-lr_cv_scores)
print((-lr_cv_scores).mean())

I used GridSearchCV with 5-fold cross-validation to search over a small grid of hyperparameters for both Random Forest and XGBoost, rather than relying on default settings.

Random Forest: best combination was n_estimators=200, max_depth=10, min_samples_split=2, improving cross-validated RMSE from 1117.26 (default) to 1091.98.
XGBoost: best combination was n_estimators=100, max_depth=3, learning_rate=0.05, achieving a cross-validated RMSE of 1081.34 — the best result of any model tested.

This showed that comparing models fairly requires tuning each one properly first — an untuned XGBoost initially looked worse than Random Forest, but its true potential only emerged after tuning.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define a small set of hyperparameter values to try
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

# Set up the grid search: try every combination, using 5-fold cross-validation, scoring by RMSE
grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

# Run it (this will take a while — it's training many models: 2 x 3 x 2 = 12 combinations, each cross-validated 5 times = 60 total model trainings)
grid_search.fit(X, y)

In [ ]:
print(grid_search.best_params_)
print(-grid_search.best_score_)

In [ ]:
sample_sub = pd.read_csv('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/sample_submission.csv')
sample_sub.head()

In [ ]:
sample_sub.columns.tolist()

In [ ]:
# Train the final Random Forest using our best tuned settings, on ALL of train (not just X_train)
final_model = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=2, random_state=42)
final_model.fit(X, y)

# Predict total_sales for the real, unseen test set
test_predictions = final_model.predict(test)

In [ ]:
# Build the submission DataFrame: id + predicted total_sales, matching sample_submission's format
submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': test_predictions
})

# Save it as a CSV file, ready to upload to Kaggle
submission.to_csv('submission.csv', index=False)

# Quick peek to confirm it looks right
submission.head()

In [ ]:
import xgboost as xgb
print(xgb.__version__)

In [ ]:
# Import the XGBoost regressor
from xgboost import XGBRegressor

# Create the model with reasonable starting settings
xgb_model = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)

# Train it on the same training split we've been using
xgb_model.fit(X_train, y_train)

# Predict on the validation set
xgb_pred = xgb_model.predict(X_val)

# Score it the same way as before
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_pred))
xgb_r2 = r2_score(y_val, xgb_pred)

print(xgb_rmse)
print(xgb_r2)

In [ ]:
# Define a small grid of XGBoost hyperparameters to try
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1]
}

# Set up grid search: try every combination, 5-fold cross-validation, scored by RMSE
xgb_grid_search = GridSearchCV(
    XGBRegressor(random_state=42),
    xgb_param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

# Run it — 2 x 3 x 2 = 12 combinations x 5 folds = 60 total trainings, similar scale to before
xgb_grid_search.fit(X, y)

In [ ]:
print(xgb_grid_search.best_params_)
print(-xgb_grid_search.best_score_)

In [ ]:
# Train the final XGBoost model using the best settings GridSearchCV found, on ALL of train
final_xgb_model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
final_xgb_model.fit(X, y)

# Predict total_sales for the real, unseen test set
xgb_test_predictions = final_xgb_model.predict(test)

In [ ]:
# Build the submission DataFrame with the new XGBoost predictions
xgb_submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': xgb_test_predictions
})

# Save it as a new CSV file (different name so we don't overwrite the Random Forest one)
xgb_submission.to_csv('submission_xgb.csv', index=False)

# Quick peek to confirm it looks right
xgb_submission.head()

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
from IPython.display import FileLink
FileLink('submission_xgb.csv')

In [ ]:
import numpy as np

# Transform the target: train on log(total_sales) instead of raw total_sales
y_log = np.log1p(y)

# Split again, this time with the log-transformed target
X_train2, X_val2, y_train_log, y_val_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

# Train XGBoost on the log-transformed target, using our best tuned settings
log_model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
log_model.fit(X_train2, y_train_log)

# Predict (still in log-scale)
log_pred = log_model.predict(X_val2)

# Reverse the transformation to get real total_sales predictions back
real_pred = np.expm1(log_pred)

# To score fairly, compare against the REAL (non-log) validation answers
y_val_real = np.expm1(y_val_log)

log_rmse = np.sqrt(mean_squared_error(y_val_real, real_pred))
print(log_rmse)

In [ ]:
# Create the new feature in both train and test
X['price_per_kg'] = X['product_price'] / X['product_weight_kg']
test['price_per_kg'] = test['product_price'] / test['product_weight_kg']

In [ ]:
# Re-split since X now has an extra column
X_train3, X_val3, y_train3, y_val3 = train_test_split(X, y, test_size=0.2, random_state=42)

# Retrain XGBoost with the new feature included
feat_model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
feat_model.fit(X_train3, y_train3)

feat_pred = feat_model.predict(X_val3)
feat_rmse = np.sqrt(mean_squared_error(y_val3, feat_pred))
print(feat_rmse)

In [ ]:
# Cross-validate XGBoost with the new price_per_kg feature included
feat_cv_scores = cross_val_score(feat_model, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(-feat_cv_scores)
print((-feat_cv_scores).mean())

In [ ]:
# Get validation predictions from both tuned models
rf_val_pred = final_model.predict(X_val)
xgb_val_pred = final_xgb_model.predict(X_val)

# Average the two sets of predictions together
ensemble_val_pred = (rf_val_pred + xgb_val_pred) / 2

# Score the ensemble the same honest way as always
ensemble_rmse = np.sqrt(mean_squared_error(y_val, ensemble_val_pred))
print(ensemble_rmse)

In [ ]:
X_val.shape
X.shape

In [ ]:
X_val.shape

In [ ]:
# Use models trained ONLY on X_train (not the full dataset) for a fair validation test
rf_train_pred = rf_model.predict(X_val)
xgb_train_pred = xgb_model.predict(X_val)

ensemble_fair_pred = (rf_train_pred + xgb_train_pred) / 2
ensemble_fair_rmse = np.sqrt(mean_squared_error(y_val, ensemble_fair_pred))
print(ensemble_fair_rmse)

In [ ]:
# Fresh tuned Random Forest, trained ONLY on X_train (not leaked)
rf_tuned_fair = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=2, random_state=42)
rf_tuned_fair.fit(X_train, y_train)

# Fresh tuned XGBoost, trained ONLY on X_train (not leaked)
xgb_tuned_fair = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
xgb_tuned_fair.fit(X_train, y_train)

# Predict on X_val with both, then average
rf_fair_pred = rf_tuned_fair.predict(X_val)
xgb_fair_pred = xgb_tuned_fair.predict(X_val)
ensemble_tuned_pred = (rf_fair_pred + xgb_fair_pred) / 2

ensemble_tuned_rmse = np.sqrt(mean_squared_error(y_val, ensemble_tuned_pred))
print(ensemble_tuned_rmse)

In [ ]:
from sklearn.model_selection import KFold

# Set up 5 folds, matching what we've used throughout
kf = KFold(n_splits=5, shuffle=True, random_state=42)

ensemble_scores = []

# Loop through each of the 5 folds
for train_idx, val_idx in kf.split(X):
    # Split X and y into this fold's train/validation portions
    X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Train fresh tuned models on this fold's training portion only
    rf_fold = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=2, random_state=42)
    rf_fold.fit(X_fold_train, y_fold_train)
    
    xgb_fold = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
    xgb_fold.fit(X_fold_train, y_fold_train)
    
    # Predict on this fold's validation portion, average the two models
    rf_fold_pred = rf_fold.predict(X_fold_val)
    xgb_fold_pred = xgb_fold.predict(X_fold_val)
    ensemble_fold_pred = (rf_fold_pred + xgb_fold_pred) / 2
    
    # Score this fold, store it
    fold_rmse = np.sqrt(mean_squared_error(y_fold_val, ensemble_fold_pred))
    ensemble_scores.append(fold_rmse)

print(ensemble_scores)
print(np.mean(ensemble_scores))

In [ ]:
# Try a few different weight combinations for XGBoost vs Random Forest
for xgb_weight in [0.6, 0.7, 0.8]:
    rf_weight = 1 - xgb_weight
    scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
        y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]
        
        rf_fold = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=2, random_state=42)
        rf_fold.fit(X_fold_train, y_fold_train)
        
        xgb_fold = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
        xgb_fold.fit(X_fold_train, y_fold_train)
        
        rf_pred = rf_fold.predict(X_fold_val)
        xgb_pred = xgb_fold.predict(X_fold_val)
        
        # Weighted blend instead of simple average
        blend_pred = (xgb_weight * xgb_pred) + (rf_weight * rf_pred)
        
        fold_rmse = np.sqrt(mean_squared_error(y_fold_val, blend_pred))
        scores.append(fold_rmse)
    
    print(f"XGBoost weight {xgb_weight}: mean RMSE = {np.mean(scores)}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# A wider range of hyperparameters to sample from
xgb_param_dist = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

# Randomly sample 30 combinations from this space, instead of trying all of them
xgb_random_search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    xgb_param_dist,
    n_iter=30,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42
)

xgb_random_search.fit(X, y)

In [ ]:
print(xgb_random_search.best_params_)
print(-xgb_random_search.best_score_)

In [ ]:
# Train the final model with the RandomizedSearchCV's best settings, on ALL of train
final_xgb_v2 = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)
final_xgb_v2.fit(X, y)

# Predict on test
xgb_v2_predictions = final_xgb_v2.predict(test)

# Build and save submission
xgb_v2_submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': xgb_v2_predictions
})
xgb_v2_submission.to_csv('submission_xgb_v2.csv', index=False)
xgb_v2_submission.head()

In [ ]:
set(X.columns) == set(test.columns)

In [ ]:
# Wider random search: try more combinations (60 instead of 30), same hyperparameter space
xgb_random_search2 = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    xgb_param_dist,
    n_iter=60,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42
)

xgb_random_search2.fit(X, y)

In [ ]:
print(xgb_random_search2.best_params_)
print(-xgb_random_search2.best_score_)

In [ ]:
import lightgbm as lgb
print(lgb.__version__)

In [ ]:
from lightgbm import LGBMRegressor

# Create the model with reasonable starting settings, matching XGBoost's scale for fair comparison
lgbm_model = LGBMRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)

# Cross-validate it the same honest way as every other model
lgbm_cv_scores = cross_val_score(lgbm_model, X, y, cv=5, scoring='neg_root_mean_squared_error')

print(-lgbm_cv_scores)
print((-lgbm_cv_scores).mean())

In [ ]:
from lightgbm import LGBMRegressor

# A wide hyperparameter space to sample from
lgbm_param_dist = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [3, 4, 5, 6, -1],  # -1 means "no limit" in LightGBM's convention
    'num_leaves': [15, 31, 50, 70],
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

lgbm_random_search = RandomizedSearchCV(
    LGBMRegressor(random_state=42, verbose=-1),
    lgbm_param_dist,
    n_iter=30,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42
)

lgbm_random_search.fit(X, y)

In [ ]:
print(lgbm_random_search.best_params_)
print(-lgbm_random_search.best_score_)

In [ ]:
X.shape
test.shape

In [ ]:
inter_cols = [col for col in X.columns if col.startswith('inter_')]
print(len(inter_cols))

In [ ]:
X.shape
test.shape

In [ ]:
X.shape

In [ ]:
# Even wider search: 80 iterations this time, slightly expanded ranges
xgb_param_dist2 = {
    'n_estimators': [50, 100, 150, 200, 300, 400],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.07, 0.1],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0]
}

xgb_random_search3 = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    xgb_param_dist2,
    n_iter=80,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42
)

xgb_random_search3.fit(X, y)

In [ ]:
print(xgb_random_search3.best_params_)
print(-xgb_random_search3.best_score_)

After settling on the tuned XGBoost model, I checked which features it actually relied on most, using its built-in `feature_importances_', a measure of how much each feature contributed to reducing error across all the model's trees. This was done to look for genuine, evidence-based directions for further feature engineering, rather than guessing blindly.

In [ ]:
importances = final_xgb_v2.feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
}).sort_values('importance', ascending=False)

print(feature_importance_df.head(15))

Reload the original store_code values temporarily, since we one-hot encoded them earlier
We can get this back from train's original store_code column before we transformed it, but since we've already transformed train, instead I'll check directly on the one-hot column itself

In [ ]:
# Average total_sales for rows where store_STORE-7WS = 1, vs everywhere else
store_7ws_sales = train.loc[train['store_STORE-7WS'] == 1, 'total_sales'].mean()
other_stores_sales = train.loc[train['store_STORE-7WS'] == 0, 'total_sales'].mean()

print(f"STORE-7WS average sales: {store_7ws_sales}")
print(f"All other stores average sales: {other_stores_sales}")

In [ ]:
train.loc[train['store_STORE-7WS'] == 1, 'store_format'].unique()

In [ ]:
import catboost
print(catboost.__version__)

In [ ]:
from catboost import CatBoostRegressor

# Create the model with reasonable starting settings, silent output to avoid clutter
catboost_model = CatBoostRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbose=0)

# Cross-validate the same honest way as every other model
catboost_cv_scores = cross_val_score(catboost_model, X, y, cv=5, scoring='neg_root_mean_squared_error')

print(-catboost_cv_scores)
print((-catboost_cv_scores).mean())

In [ ]:
# CatBoost-specific hyperparameter space
catboost_param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7, 9]
}

catboost_random_search = RandomizedSearchCV(
    CatBoostRegressor(random_state=42, verbose=0),
    catboost_param_dist,
    n_iter=30,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42
)

catboost_random_search.fit(X, y)

In [ ]:
print(catboost_random_search.best_params_)
print(-catboost_random_search.best_score_)

In [ ]:
# Train final CatBoost model with best settings, on ALL of train
final_catboost = CatBoostRegressor(
    n_estimators=100, max_depth=5, learning_rate=0.07, l2_leaf_reg=1, random_state=42, verbose=0
)
final_catboost.fit(X, y)

# Predict on test
catboost_predictions = final_catboost.predict(test)

# Build and save submission
catboost_submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': catboost_predictions
})
catboost_submission.to_csv('submission_catboost.csv', index=False)
catboost_submission.head()

In [ ]:
import os
print(os.listdir('/kaggle/working'))

In [ ]:
# Basic stats on shelf_visibility
print(train['shelf_visibility'].describe())

Early EDA found a weak negative correlation (-0.12) between shelf_visibility and total_sales — counter-intuitive, since higher shelf visibility would be expected to help sales. I investigated whether this was distorted by a data quality issue: 422 rows (~6.2%) had shelf_visibility recorded as exactly 0, and I checked whether these might represent disguised missing values rather than genuine "no visibility" cases (similar to how store_size and product_weight_kg had missing values). Comparing average sales for zero-visibility rows (2154.62) against non-zero rows (2176.09) showed no meaningful difference, ruling out this explanation. The weak negative correlation appears to be a genuine, if minor, characteristic of the data rather than an artifact — no further action was taken on this feature.

In [ ]:
# How many rows have exactly zero shelf_visibility?
(train['shelf_visibility'] == 0).sum()

In [ ]:
# Average sales for zero-visibility rows vs everything else
zero_vis_sales = train.loc[train['shelf_visibility'] == 0, 'total_sales'].mean()
nonzero_vis_sales = train.loc[train['shelf_visibility'] > 0, 'total_sales'].mean()

print(f"Zero visibility average sales: {zero_vis_sales}")
print(f"Non-zero visibility average sales: {nonzero_vis_sales}")


<!-- ?;'After testing four different algorithms (Linear Regression, Random Forest, XGBoost, LightGBM, and CatBoost), multiple rounds of hyperparameter tuning, ensembling, and two feature engineering attempts (price_per_kg and a store_format × product_category interaction), the final model remained: XGBoost with n_estimators=300, max_depth=2, learning_rate=0.03, subsample=0.9, colsample_bytree=0.9, trained on the full training set, achieving a leaderboard RMSE of 1074.74. Feature importance analysis confirmed the model h/????////ad correctly identified the dataset's main learnable patterns (price, one exceptionally high-performing store, and category differences), and further experimentation showed diminishing and eventually unreliable returns — a reasonable, evidence-based stopping point for this iteration of the project. -->

In [ ]:
# Retrain our best XGBoost on X_train only (fair test on X_val, not leaked)
best_xgb = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)
best_xgb.fit(X_train, y_train)

# Predict on validation set
val_preds = best_xgb.predict(X_val)

# Calculate residuals
residuals = y_val.values - val_preds

# Build a DataFrame combining actual, predicted, residual, and the original features for inspection
residual_df = X_val.copy()
residual_df['actual'] = y_val.values
residual_df['predicted'] = val_preds
residual_df['residual'] = residuals
residual_df['abs_residual'] = np.abs(residuals)

# Look at the 10 WORST predictions (biggest absolute errors)
worst_10 = residual_df.sort_values('abs_residual', ascending=False).head(10)
print(worst_10[['actual', 'predicted', 'residual', 'product_price', 'store_format', 'store_size']])

In [ ]:
# Check: among high-price products, how much does actual total_sales vary?
high_price_rows = X_val[X_val['product_price'] > 200].copy()
high_price_rows['actual'] = y_val.values[X_val['product_price'].values > 200]

print(high_price_rows['actual'].describe())

In [ ]:
# Within high-price products, does store_format explain the sales spread?
high_price_rows.groupby('store_format')['actual'].agg(['mean', 'std', 'count'])

In [ ]:
# Create a price × store_format interaction feature (numeric multiplication, not one-hot)
X['price_x_format'] = X['product_price'] * X['store_format']
test['price_x_format'] = test['product_price'] * test['store_format']

In [ ]:
X.shape
test.shape

In [ ]:
X.shape

In [ ]:
# Retrain XGBoost with best known settings, now including price_x_format
xgb_pricefmt_model = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)

pricefmt_cv_scores = cross_val_score(xgb_pricefmt_model, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(-pricefmt_cv_scores)
print((-pricefmt_cv_scores).mean())

In [ ]:
# Quick check: does this feature's benefit hold up with fresh tuning, or was 1078.76 already close to its ceiling?
xgb_pricefmt_search = RandomizedSearchCV(
    XGBRegressor(random_state=42),
    xgb_param_dist2,
    n_iter=40,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42
)
xgb_pricefmt_search.fit(X, y)

print(xgb_pricefmt_search.best_params_)
print(-xgb_pricefmt_search.best_score_)

In [ ]:
# Train final model with the price_x_format feature, using original best settings, on ALL of train
final_xgb_pricefmt = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)
final_xgb_pricefmt.fit(X, y)

# Predict on test
pricefmt_predictions = final_xgb_pricefmt.predict(test)

# Build and save submission
pricefmt_submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': pricefmt_predictions
})
pricefmt_submission.to_csv('submission_pricefmt.csv', index=False)
pricefmt_submission.head()

In [ ]:
# Retrain on X_train only (fair test), now including price_x_format
best_xgb_v2 = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)
best_xgb_v2.fit(X_train, y_train)

# Note: X_train/X_val were split BEFORE we added price_x_format, so we need to rebuild them
# Let's do a fresh split on the current X (which now includes price_x_format)
X_train2, X_val2, y_train2, y_val2 = train_test_split(X, y, test_size=0.2, random_state=42)

best_xgb_v2.fit(X_train2, y_train2)
val_preds2 = best_xgb_v2.predict(X_val2)

residuals2 = y_val2.values - val_preds2
residual_df2 = X_val2.copy()
residual_df2['actual'] = y_val2.values
residual_df2['predicted'] = val_preds2
residual_df2['residual'] = residuals2
residual_df2['abs_residual'] = np.abs(residuals2)

worst_10_v2 = residual_df2.sort_values('abs_residual', ascending=False).head(10)
print(worst_10_v2[['actual', 'predicted', 'residual', 'product_price', 'store_format', 'store_size', 'store_location_tier']])

In [ ]:
# Check store_size and store_location_tier distribution among ALL high-price, high-actual-sales rows
extreme_high_sales = residual_df2[(residual_df2['product_price'] > 150) & (residual_df2['actual'] > 7000)]
print(extreme_high_sales[['store_size', 'store_location_tier', 'store_format']].value_counts())
print(f"\nTotal such rows: {len(extreme_high_sales)}")

In [ ]:
train.loc[train['store_STORE-7WS'] == 1, ['store_size', 'store_location_tier']].drop_duplicates()

In [ ]:
# Reconstruct original store_code from the one-hot store_ columns
store_cols = [col for col in X.columns if col.startswith('store_STORE')]
X['store_code_temp'] = X[store_cols].idxmax(axis=1).str.replace('store_', '')
test['store_code_temp'] = test[store_cols].idxmax(axis=1).str.replace('store_', '')

print(X['store_code_temp'].unique())

In [ ]:
from sklearn.model_selection import KFold

# Initialize an empty column to hold the encoded values
X['store_target_enc'] = 0.0

# Use 5-fold splitting, same setup as before
kf_encode = KFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in kf_encode.split(X):
    # Calculate each store's average sales, using ONLY this fold's training portion
    fold_train_data = X.iloc[train_idx].copy()
    fold_train_data['total_sales_temp'] = y.iloc[train_idx].values
    
    store_means = fold_train_data.groupby('store_code_temp')['total_sales_temp'].mean()
    
    # Apply those means to the VALIDATION portion of this fold only
    X.loc[X.index[val_idx], 'store_target_enc'] = X.iloc[val_idx]['store_code_temp'].map(store_means).values

# For test set: use the FULL train data's store means (no leakage risk here, since test has no y at all)
full_store_means = X.groupby('store_code_temp').apply(lambda g: y.loc[g.index].mean())
test['store_target_enc'] = test['store_code_temp'].map(full_store_means)

print(X[['store_code_temp', 'store_target_enc']].head(10))

In [ ]:
X['total_sales_check'] = y.values
X.groupby('store_code_temp')['total_sales_check'].mean()

In [ ]:
# Remove the temporary check column
X = X.drop(columns=['total_sales_check'])

# Retrain and cross-validate with the new store_target_enc feature included
xgb_targetenc_model = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)

# Use only numeric columns (drop the temporary text column before training)
X_for_model = X.drop(columns=['store_code_temp'])

targetenc_cv_scores = cross_val_score(xgb_targetenc_model, X_for_model, y, cv=5, scoring='neg_root_mean_squared_error')
print(-targetenc_cv_scores)
print((-targetenc_cv_scores).mean())

In [ ]:
final_xgb_targetenc = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)

X_full_for_model = X.drop(columns=['store_code_temp'])
final_xgb_targetenc.fit(X_full_for_model, y)

test_for_model = test.drop(columns=['store_code_temp'])
targetenc_predictions = final_xgb_targetenc.predict(test_for_model)

targetenc_submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': targetenc_predictions
})
targetenc_submission.to_csv('submission_targetenc.csv', index=False)
targetenc_submission.head()

In [ ]:
from IPython.display import FileLink
FileLink('submission_targetenc.csv')

In [ ]:
from sklearn.model_selection import KFold

X_stack = X.drop(columns=['store_code_temp'])

kf_stack = KFold(n_splits=5, shuffle=True, random_state=42)

# Empty arrays to hold out-of-fold predictions from each base model
oof_xgb = np.zeros(len(X_stack))
oof_rf = np.zeros(len(X_stack))
oof_catboost = np.zeros(len(X_stack))

for train_idx, val_idx in kf_stack.split(X_stack):
    X_fold_train, X_fold_val = X_stack.iloc[train_idx], X_stack.iloc[val_idx]
    y_fold_train = y.iloc[train_idx]
    
    xgb_s = XGBRegressor(subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42)
    xgb_s.fit(X_fold_train, y_fold_train)
    oof_xgb[val_idx] = xgb_s.predict(X_fold_val)
    
    rf_s = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=2, random_state=42)
    rf_s.fit(X_fold_train, y_fold_train)
    oof_rf[val_idx] = rf_s.predict(X_fold_val)
    
    cat_s = CatBoostRegressor(n_estimators=100, max_depth=5, learning_rate=0.07, l2_leaf_reg=1, random_state=42, verbose=0)
    cat_s.fit(X_fold_train, y_fold_train)
    oof_catboost[val_idx] = cat_s.predict(X_fold_val)

print("Done generating out-of-fold predictions")

In [ ]:
# Build a small DataFrame: each base model's out-of-fold predictions as "features"
stack_features = pd.DataFrame({
    'xgb_pred': oof_xgb,
    'rf_pred': oof_rf,
    'catboost_pred': oof_catboost
})

# Use a simple Linear Regression as the meta-model — a common, sensible default for stacking
from sklearn.linear_model import LinearRegression

meta_model = LinearRegression()

# Cross-validate the meta-model itself, to see how well this combination performs
meta_cv_scores = cross_val_score(meta_model, stack_features, y, cv=5, scoring='neg_root_mean_squared_error')
print(-meta_cv_scores)
print((-meta_cv_scores).mean())

In [ ]:
print(y.index[:10])
print(stack_features.index[:10])

In [ ]:
# Check each base model's own out-of-fold RMSE individually
print("XGBoost OOF RMSE:", np.sqrt(mean_squared_error(y, oof_xgb)))
print("Random Forest OOF RMSE:", np.sqrt(mean_squared_error(y, oof_rf)))
print("CatBoost OOF RMSE:", np.sqrt(mean_squared_error(y, oof_catboost)))

In [ ]:
X_stack.isnull().sum().sum()

In [ ]:
print(len(X_stack))
print(len(y))
print(X_stack.shape)

In [ ]:
print(oof_xgb[:5])
print(y.values[:5])

In [ ]:
print((oof_xgb != 0).sum())

In [ ]:
X_stack = X.drop(columns=['store_code_temp'])

oof_xgb = np.zeros(len(X_stack))
oof_rf = np.zeros(len(X_stack))
oof_catboost = np.zeros(len(X_stack))

kf_stack = KFold(n_splits=5, shuffle=True, random_state=42)

fold_num = 0
for train_idx, val_idx in kf_stack.split(X_stack):
    fold_num += 1
    print(f"Fold {fold_num}: train size = {len(train_idx)}, val size = {len(val_idx)}")
    
    X_fold_train, X_fold_val = X_stack.iloc[train_idx], X_stack.iloc[val_idx]
    y_fold_train = y.iloc[train_idx]
    
    xgb_s = XGBRegressor(subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42)
    xgb_s.fit(X_fold_train, y_fold_train)
    oof_xgb[val_idx] = xgb_s.predict(X_fold_val)
    
    rf_s = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=2, random_state=42)
    rf_s.fit(X_fold_train, y_fold_train)
    oof_rf[val_idx] = rf_s.predict(X_fold_val)
    
    cat_s = CatBoostRegressor(n_estimators=100, max_depth=5, learning_rate=0.07, l2_leaf_reg=1, random_state=42, verbose=0)
    cat_s.fit(X_fold_train, y_fold_train)
    oof_catboost[val_idx] = cat_s.predict(X_fold_val)
    
    print(f"Fold {fold_num} done. Non-zero count so far: {(oof_xgb != 0).sum()}")

print("All folds complete")
print((oof_xgb != 0).sum())

In [ ]:
print("XGBoost OOF RMSE:", np.sqrt(mean_squared_error(y, oof_xgb)))
print("Random Forest OOF RMSE:", np.sqrt(mean_squared_error(y, oof_rf)))
print("CatBoost OOF RMSE:", np.sqrt(mean_squared_error(y, oof_catboost)))

In [ ]:
# Load a fresh, untouched copy directly from the original file, just for this check
train_original = pd.read_csv('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/train.csv')

product_sales_check = train_original.groupby('product_code')['total_sales'].agg(['mean', 'count'])
print(product_sales_check['mean'].describe())
print(product_sales_check['count'].describe())

In [ ]:
# We need product_code back in our working X - let's add it fresh from the original data
X['product_code_temp'] = train_original['product_code'].values

smoothing_factor_product = 20  # higher than store's, since groups are much smaller here

X['product_target_enc'] = 0.0
global_mean = y.mean()

for train_idx, val_idx in kf_encode.split(X):
    fold_train_data = X.iloc[train_idx].copy()
    fold_train_data['total_sales_temp'] = y.iloc[train_idx].values
    
    prod_stats = fold_train_data.groupby('product_code_temp')['total_sales_temp'].agg(['mean', 'count'])
    prod_stats['smoothed'] = (prod_stats['count'] * prod_stats['mean'] + smoothing_factor_product * global_mean) / (prod_stats['count'] + smoothing_factor_product)
    
    X.loc[X.index[val_idx], 'product_target_enc'] = X.iloc[val_idx]['product_code_temp'].map(prod_stats['smoothed']).values

print(X[['product_code_temp', 'product_target_enc']].head(10))

In [ ]:
# Build test's product target encoding using ALL of train's data (no leakage risk, same as before)
test_original = pd.read_csv('/kaggle/input/competitions/dsn-bootcamp-qualification-hackathon-2026-ml-track/test.csv')
test['product_code_temp'] = test_original['product_code'].values

full_prod_stats = X.groupby('product_code_temp').apply(lambda g: y.loc[g.index].mean())
full_prod_counts = X.groupby('product_code_temp').size()
full_prod_smoothed = (full_prod_counts * full_prod_stats + smoothing_factor_product * global_mean) / (full_prod_counts + smoothing_factor_product)

test['product_target_enc'] = test['product_code_temp'].map(full_prod_smoothed)

# IMPORTANT: some test products might not exist in train at all - check for missing values
print(test['product_target_enc'].isnull().sum())

In [ ]:
# Fill any missing product encodings with the global mean (a safe, neutral default)
test['product_target_enc'] = test['product_target_enc'].fillna(global_mean)

print(test['product_target_enc'].isnull().sum())

In [ ]:
print('X_reduced' in dir())
print('X_smooth_test' in dir())
print('product_target_enc' in X.columns)

In [ ]:
# Rebuild X_reduced: our full feature set minus the zero-importance features found earlier
zero_importance_features = ['fat_content', 'cat_dairy', 'cat_baking goods', 'cat_health and hygiene', 
                              'cat_others', 'cat_household', 'cat_frozen foods', 'store_STORE-89Z', 
                              'store_STORE-T5G', 'store_STORE-OYG', 'store_STORE-JOR', 'store_STORE-YLW']

X_reduced = X.drop(columns=[c for c in zero_importance_features if c in X.columns])

# Add price_bin
X_reduced['price_bin'] = pd.qcut(X_reduced['product_price'], q=5, labels=False)

print(X_reduced.shape)
print(X_reduced.columns.tolist())

In [ ]:
smoothing_factor = 10
X_reduced['store_target_enc_smooth'] = 0.0
global_mean = y.mean()

for train_idx, val_idx in kf_encode.split(X_reduced):
    fold_train_data = X_reduced.iloc[train_idx].copy()
    fold_train_data['total_sales_temp'] = y.iloc[train_idx].values
    
    store_stats = fold_train_data.groupby('store_code_temp')['total_sales_temp'].agg(['mean', 'count'])
    store_stats['smoothed'] = (store_stats['count'] * store_stats['mean'] + smoothing_factor * global_mean) / (store_stats['count'] + smoothing_factor)
    
    X_reduced.loc[X_reduced.index[val_idx], 'store_target_enc_smooth'] = X_reduced.iloc[val_idx]['store_code_temp'].map(store_stats['smoothed']).values

# Now build X_smooth_test: drop the raw/temp columns, keep the smoothed version
X_smooth_test = X_reduced.drop(columns=['store_code_temp', 'store_target_enc'])

print(X_smooth_test.shape)

In [ ]:
X_with_product = X_smooth_test.drop(columns=['product_code_temp']).copy()

xgb_product_test = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)

product_cv_scores = cross_val_score(xgb_product_test, X_with_product, y, cv=5, scoring='neg_root_mean_squared_error')
print(-product_cv_scores)
print((-product_cv_scores).mean())

In [ ]:
# Rebuild test_reduced, matching X_reduced's construction
test_reduced = test.drop(columns=[c for c in zero_importance_features if c in test.columns])
test_reduced['price_bin'] = pd.qcut(test_reduced['product_price'], q=5, labels=False)

print(test_reduced.shape)
print(test_reduced.columns.tolist())

In [ ]:
# Rebuild test's smoothed store encoding
full_store_stats = X_reduced.groupby('store_code_temp').apply(lambda g: y.loc[g.index].mean())
full_store_counts = X_reduced.groupby('store_code_temp').size()
full_smoothed_store = (full_store_counts * full_store_stats + smoothing_factor * global_mean) / (full_store_counts + smoothing_factor)
test_reduced['store_target_enc_smooth'] = test_reduced['store_code_temp'].map(full_smoothed_store)

# Build final test feature set, matching X_with_product's columns exactly
test_final = test_reduced.drop(columns=['store_code_temp', 'store_target_enc', 'product_code_temp'], errors='ignore').copy()

print(set(X_with_product.columns) == set(test_final.columns))

In [ ]:
print('X_with_product' in dir())
print('final_xgb_product' in dir())

In [ ]:
final_xgb_product = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)
final_xgb_product.fit(X_with_product, y)

In [ ]:
product_predictions = final_xgb_product.predict(test_final)

product_submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': product_predictions
})
product_submission.to_csv('submission_product.csv', index=False)
product_submission.head()

In [ ]:
# Test a few different smoothing factors, see which gives the best cross-validated result
for smooth_val in [5, 10, 15, 20, 30]:
    X_temp = X_reduced.drop(columns=['store_code_temp', 'store_target_enc', 'product_code_temp'], errors='ignore').copy()
    
    prod_enc_temp = pd.Series(0.0, index=X.index)
    for train_idx, val_idx in kf_encode.split(X):
        fold_train_data = X.iloc[train_idx].copy()
        fold_train_data['total_sales_temp'] = y.iloc[train_idx].values
        prod_stats = fold_train_data.groupby('product_code_temp')['total_sales_temp'].agg(['mean', 'count'])
        prod_stats['smoothed'] = (prod_stats['count'] * prod_stats['mean'] + smooth_val * global_mean) / (prod_stats['count'] + smooth_val)
        prod_enc_temp.iloc[val_idx] = X.iloc[val_idx]['product_code_temp'].map(prod_stats['smoothed']).values
    
    X_temp['product_target_enc'] = prod_enc_temp.values
    
    model_temp = XGBRegressor(subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42)
    scores_temp = cross_val_score(model_temp, X_temp, y, cv=5, scoring='neg_root_mean_squared_error')
    print(f"Smoothing={smooth_val}: mean RMSE = {-scores_temp.mean()}")

In [ ]:
X_prod_store = X_temp.copy()  # reuse the version with smoothing=30 from the loop, or rebuild cleanly with 20

# Rebuild cleanly with our confirmed best smoothing=20 for product encoding
X_prod_store = X_reduced.drop(columns=['store_code_temp', 'store_target_enc', 'product_code_temp'], errors='ignore').copy()

# Create the interaction
X_prod_store['product_x_store'] = X_prod_store['product_target_enc'] * X_prod_store['store_target_enc_smooth']

print(X_prod_store[['product_target_enc', 'store_target_enc_smooth', 'product_x_store']].head())

In [ ]:
xgb_prodstore = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)

prodstore_cv_scores = cross_val_score(xgb_prodstore, X_prod_store, y, cv=5, scoring='neg_root_mean_squared_error')
print(-prodstore_cv_scores)
print((-prodstore_cv_scores).mean())

In [ ]:
final_xgb_prodstore = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)
final_xgb_prodstore.fit(X_prod_store, y)

# Build matching test set with the same interaction
test_prod_store = test_final.copy()
test_prod_store['product_x_store'] = test_prod_store['product_target_enc'] * test_prod_store['store_target_enc_smooth']

print(set(X_prod_store.columns) == set(test_prod_store.columns))

In [ ]:
prodstore_predictions = final_xgb_prodstore.predict(test_prod_store)

prodstore_submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': prodstore_predictions
})
prodstore_submission.to_csv('submission_prodstore.csv', index=False)
prodstore_submission.head()

In [ ]:
X_full_interact = X_prod_store.copy()
X_full_interact['price_x_product'] = X_full_interact['product_price'] * X_full_interact['product_target_enc']

xgb_full_interact = XGBRegressor(
    subsample=0.9, n_estimators=300, max_depth=2, learning_rate=0.03, colsample_bytree=0.9, random_state=42
)

full_interact_scores = cross_val_score(xgb_full_interact, X_full_interact, y, cv=5, scoring='neg_root_mean_squared_error')
print(-full_interact_scores)
print((-full_interact_scores).mean())

In [ ]:
# CatBoost on current best features (X_prod_store, our confirmed 1069.91 feature set)
catboost_retest = CatBoostRegressor(n_estimators=100, max_depth=5, learning_rate=0.07, l2_leaf_reg=1, random_state=42, verbose=0)
catboost_retest_scores = cross_val_score(catboost_retest, X_prod_store, y, cv=5, scoring='neg_root_mean_squared_error')
print("CatBoost:", -catboost_retest_scores.mean())

# LightGBM on the same feature set
lgbm_retest = LGBMRegressor(subsample=1.0, num_leaves=50, n_estimators=50, max_depth=3, learning_rate=0.1, colsample_bytree=0.9, random_state=42, verbose=-1)
lgbm_retest_scores = cross_val_score(lgbm_retest, X_prod_store, y, cv=5, scoring='neg_root_mean_squared_error')
print("LightGBM:", -lgbm_retest_scores.mean())

In [ ]:
import optuna

def catboost_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
    }
    
    model = CatBoostRegressor(**params, random_state=42, verbose=0)
    scores = cross_val_score(model, X_prod_store, y, cv=5, scoring='neg_root_mean_squared_error')
    return -scores.mean()

catboost_study = optuna.create_study(direction='minimize')
catboost_study.optimize(catboost_objective, n_trials=50)

print(catboost_study.best_params)
print(catboost_study.best_value)

In [ ]:
final_catboost_v2 = CatBoostRegressor(
    n_estimators=361, max_depth=6, learning_rate=0.022401773781912146, l2_leaf_reg=7.3784950272953695,
    random_state=42, verbose=0
)
final_catboost_v2.fit(X_prod_store, y)

catboost_v2_predictions = final_catboost_v2.predict(test_prod_store)

catboost_v2_submission = pd.DataFrame({
    'id': test_ids,
    'total_sales': catboost_v2_predictions
})
catboost_v2_submission.to_csv('submission_catboost_v2.csv', index=False)
catboost_v2_submission.head()